# Project 1: Next Word Predictor (Language Modeling with LSTM/GRU)

This project builds a **Next Word Prediction (Language Model)** using real text data (Tiny Shakespeare / Gutenberg Real Text Corpus) in PyTorch.

---

## Key Topics Covered:

1. **Real Text Data Pipeline**: Downloading real text corpus, cleaning, word tokenization, and building `word2idx` / `idx2word` vocabularies.
2. **N-gram / Sliding Window Sequence Creation**: Converting unstructured continuous text into supervised input-target word sequence pairs ($x_{1:N} \rightarrow y$).
3. **LSTM / GRU Language Model Architecture**: `nn.Embedding` $\rightarrow$ Multi-layer Stacked `nn.LSTM` / `nn.GRU` $\rightarrow$ `nn.Dropout` $\rightarrow$ `nn.Linear` vocabulary projection.
4. **Training & Loss Optimization**: Categorical Cross-Entropy loss over total vocabulary items.
5. **Autoregressive Text Generation**: Generating new text using temperature sampling ($T=0.7, 1.0$) and top-$k$ / top-$p$ nucleus sampling.


In [4]:
import os
import re
import urllib.request
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using PyTorch Version: {torch.__version__}")
print(f"Target Computing Device: {device}")


Using PyTorch Version: 2.11.0+cpu
Target Computing Device: cpu


## 1. Real Text Dataset Ingestion & Tokenization

We download the real **Tiny Shakespeare** text dataset (or load any raw `.txt` file) and perform word tokenization and vocabulary construction.


In [5]:
# Download Real Text Dataset (Tiny Shakespeare)
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = "tinyshakespeare.txt"

if not os.path.exists(DATA_PATH):
    print("Downloading real text corpus...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("Download completed.")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read(50000) # Use first 50,000 characters for demo training

print(f"Loaded Real Text Corpus (Length: {len(raw_text):,} characters)")
print("First 200 characters sample:\n", raw_text[:200])


Download completed.
Loaded Real Text Corpus (Length: 50,000 characters)
First 200 characters sample:
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [6]:
# Text Cleaning and Tokenization
def clean_and_tokenize(text):
    # Lowercase and keep letters, numbers, basic punctuation
    text = text.lower()
    tokens = re.findall(r"\w+|[^\w\s]", text)
    return tokens

tokens = clean_and_tokenize(raw_text)
vocab = sorted(list(set(tokens)))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}

vocab_size = len(vocab)
print(f"Total Word Tokens: {len(tokens):,}")
print(f"Unique Vocabulary Size: {vocab_size:,}")


Total Word Tokens: 11,644
Unique Vocabulary Size: 2,005


## 2. Sliding Window Dataset Creation for PyTorch

We construct sequence pairs where the input is a sequence of $N$ consecutive words, and the target is the $(N+1)$-th next word.


In [7]:
class NextWordDataset(Dataset):
    def __init__(self, tokens, word2idx, seq_length=5):
        self.seq_length = seq_length
        self.inputs = []
        self.targets = []
        
        token_indices = [word2idx[t] for t in tokens]
        
        for i in range(len(token_indices) - seq_length):
            self.inputs.append(token_indices[i : i + seq_length])
            self.targets.append(token_indices[i + seq_length])
            
        self.inputs = torch.tensor(self.inputs, dtype=torch.long)
        self.targets = torch.tensor(self.targets, dtype=torch.long)
        
    def __len__(self):
        return len(self.inputs)
        
    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

seq_length = 5
dataset = NextWordDataset(tokens, word2idx, seq_length=seq_length)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

print(f"Total Sequence Examples: {len(dataset):,}")
print(f"Sample Input (Indices): {dataset[0][0].numpy()} -> Target: {dataset[0][1].item()}")
print(f"Sample Words: {[idx2word[i] for i in dataset[0][0].numpy()]} -> Next: '{idx2word[dataset[0][1].item()]}'")


Total Sequence Examples: 11,639
Sample Input (Indices): [ 648  310    5  150 1904] -> Target: 1346
Sample Words: ['first', 'citizen', ':', 'before', 'we'] -> Next: 'proceed'


## 3. Next Word Predictor Neural Architecture (LSTM/GRU)


In [8]:
class NextWordLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            embed_dim, 
            hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, hidden=None):
        embeds = self.embedding(x) # (B, Seq_Len, Embed_Dim)
        out, hidden = self.lstm(embeds, hidden) # out: (B, Seq_Len, Hidden_Dim)
        last_out = out[:, -1, :] # Take output at final time step: (B, Hidden_Dim)
        logits = self.fc(last_out) # (B, Vocab_Size)
        return logits, hidden

model = NextWordLSTM(vocab_size=vocab_size, embed_dim=128, hidden_dim=256, num_layers=2).to(device)
print(model)


NextWordLSTM(
  (embedding): Embedding(2005, 128)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=256, out_features=2005, bias=True)
)


## 4. Model Training & Loss Optimization


In [9]:
def train_next_word_model(model, train_loader, epochs=10, lr=1e-3):
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    losses = []
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            logits, _ = model(inputs)
            loss = criterion(logits, targets)
            loss.backward()
            
            # Clip gradients to prevent exploding gradients in LSTMs
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            
        epoch_loss = running_loss / len(train_loader.dataset)
        losses.append(epoch_loss)
        print(f"Epoch [{epoch+1:02d}/{epochs:02d}] - Loss: {epoch_loss:.4f}")
        
    return losses

# Uncomment line below to execute training:
# loss_history = train_next_word_model(model, train_loader, epochs=5)


## 5. Autoregressive Text Generation with Temperature & Top-K Sampling


In [10]:
def generate_next_words(model, prompt, num_words=15, temperature=0.8, top_k=5):
    model.eval()
    model.to("cpu")
    
    prompt_tokens = clean_and_tokenize(prompt)
    if len(prompt_tokens) < seq_length:
        print(f"Prompt must contain at least {seq_length} words.")
        return prompt
        
    generated = list(prompt_tokens)
    
    with torch.no_grad():
        for _ in range(num_words):
            # Take last seq_length tokens
            input_words = generated[-seq_length:]
            input_ids = [word2idx.get(w, 0) for w in input_words]
            input_tensor = torch.tensor([input_ids], dtype=torch.long)
            
            logits, _ = model(input_tensor)
            logits = logits.squeeze(0) / temperature
            
            # Top-K Filtering
            top_k_logits, top_k_indices = torch.topk(logits, top_k)
            probs = F.softmax(top_k_logits, dim=-1)
            
            # Sample next word index from top-k distribution
            next_idx_in_topk = torch.multinomial(probs, num_samples=1).item()
            next_word_idx = top_k_indices[next_idx_in_topk].item()
            next_word = idx2word[next_word_idx]
            
            generated.append(next_word)
            
    return " ".join(generated)

# Sample generation demonstration
sample_prompt = "first citizen : before we proceed"
print("Prompt:", sample_prompt)
print("Generated Text (Demo):", generate_next_words(model, sample_prompt, num_words=10, temperature=0.7, top_k=5))


Prompt: first citizen : before we proceed
Generated Text (Demo): first citizen : before we proceed conversation because mine mine chests meet our mine chests advance
